# Neural network self portrait

First we simply import the used numpy and pandas packages, as well as the file for generating animations and connecting to the neural network.

In [1]:
import numpy as np
import pandas as pd
import animations as anim

Now we can generate the animations; after that, their management will be executed completely automatically.

In [2]:
anim.generate()

After that, we set the seed for random functions to 42. We could have used any arbitrary value, but in the programming community this one is typically used, which is why we chose it too.

In [3]:
np.random.seed(42)

Now we can import the files for training and testing the neural network.

In [4]:
train = pd.read_csv("data/cs-training.csv")
test = pd.read_csv("data/cs-test.csv")

Here we remove the Id column, as it brings no useful information to the network.

In [5]:
train = train.drop(columns = ["Id"])
test = test.drop(columns = ["Id"])

In the next cell, we look for NA (Not Available) values and replace them with the median of the corresponding column from the train dataset.

In [6]:
for col in ["MonthlyIncome", "NumberOfDependents"]:
    train_median = train[col].median()
    train[col] = train[col].fillna(train_median)
    test[col] = test[col].fillna(train_median)

After that, we create a new column with the value of the monthly debt by multiplying the debt-to-income ratio by the monthly income. We do this because more intuitive parameters can improve the network's accuracy.

In [7]:
train["TotalDebt"] = train["DebtRatio"]*train["MonthlyIncome"]
test["TotalDebt"] = test["DebtRatio"]*test["MonthlyIncome"]

Now we can change the scale from linear to logarithmic for columns where such a distribution of values predominates. In this way, the model can more easily recognize and evaluate differences between lower values, which represent the majority of the data. On a linear scale, their relative differences are harder to detect due to the presence of very high values.

In [8]:
for col in ["MonthlyIncome", "DebtRatio"]:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

In the next cell, we cap the maximum values at 20 for columns representing the number of delays for a specific time period. This is because the data contains higher values (96 and 98) which have a special meaning, usually tied to a large delay or even credit default. Reducing these values to 20 maintains the negative connotation, but, similar to the previous case, increases the relative difference between lower values and thus improves the network's accuracy.

In [9]:
for col in ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]:
    train[col] = np.clip(train[col], 0, 20)
    test[col] = np.clip(test[col], 0, 20)

Subsequently, we limit the maximum ratio between used funds and their limit to 1, as values above this threshold are meaningless.

In [10]:
col = "RevolvingUtilizationOfUnsecuredLines"
train[col] = np.clip(train[col], 0, 1)
test[col] = np.clip(test[col], 0, 1)

After that, we define the input X and output y based on the train dataset. X is an 11-dimensional vector, and y is a Boolean value that indicates whether a credit default occurred within a 2-year period.

In [11]:
X = train.drop("SeriousDlqin2yrs", axis = 1).values
y = train["SeriousDlqin2yrs"].values.reshape(-1, 1)

Here we create indices for the input vectors, shuffle them, and divide them into two sets in an 80%-20% ratio. The first will serve for training the network, and the second for its validation. The accuracy of the network must be the same whether using the first or the second set. When the network's accuracy improves on the training set but worsens on the validation set as the number of learning epochs increases, it means the network is overfitted to the used data, which is of course undesirable.

In [12]:
indices = np.random.permutation(len(X))
split = int(0.8*len(X))
train_idx = indices[:split]
val_idx = indices[split:]

Now we shuffle the input and output values so they match the indices from the previous cell.

In [13]:
X_train = X[train_idx]
y_train = y[train_idx]
X_val = X[val_idx]
y_val = y[val_idx]

After that, we normalize the training and validation sets based on the mean and standard deviation of the data from each column of the training set, which naturally corresponds to each of the 11 dimensions of the input vector.

In [14]:
X_mean = X_train.mean(axis = 0)
X_std = X_train.std(axis = 0)
X_train = (X_train - X_mean)/X_std
X_val = (X_val - X_mean)/X_std

In the next cell, we prepare a vector through which we will assign a higher weight to the less frequent output value and, similarly, a lower weight to the more frequent output value.

In [15]:
classes, counts = np.unique(y, return_counts = True)
total = len(y)
class_weights = {c: total/(2*count) for c, count in zip(classes, counts)}
sample_weights = np.array([class_weights[yi[0]] for yi in y]).reshape(-1, 1)

Here we finally define the architecture of the neural network: it will have as many input values as there are dimensions in the input vector X, 3 hidden layers with 8 neurons, and 1 output value, which will provide the probability of a credit default occurring within 2 years.

In [16]:
layers = [X_train.shape[1], 8, 8, 8, 1]

Now we can initialize the values of all weights between neurons from adjacent layers and the bias values of individual neurons. For the weights, we use He initialization, while we simply set the bias values to 0.

In [17]:
weights = []
biases = []
for i in range(len(layers) - 1):
    w = np.random.randn(layers[i], layers[i + 1])*np.sqrt(2/layers[i])
    b = np.zeros((1, layers[i + 1]))
    weights.append(w)
    biases.append(b)

After that, we define the sigmoid activation function; we need it in the step from the penultimate layer to the output value, and it is therefore the one that returns the probability of credit default in the aforementioned timeframe.

In [18]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))

In the next cell, we define the ReLU (Rectified Linear Unit) activation function; we use it in the step from the input vector to the first layer of neurons, and in every subsequent step except the last one, where, as already mentioned, we use the sigmoid function. At this point, it is worth mentioning that we used He initialization in the 17th cell because it is the most suitable for networks that operate based on the ReLU activation function.

In [19]:
def relu(x):
    return np.maximum(0, x)

Finally, we define the Heaviside activation function, taking into account the unconventional H(0)=0 since it actually represents the derivative of the ReLU activation function. We use the Heaviside function during the backpropagation of weights and biases.

In [20]:
def heaviside(x):
    return (x > 0).astype(float)

Subsequently, we define a function for evaluating the probability of credit default within 2 years. This function naturally takes the input vector and, through weights, biases, and the sigmoid and ReLU activation functions, calculates and returns the output value.

In [21]:
def prob(x):
    a = x
    for i in range(len(weights) - 1):
        a = relu(np.dot(a, weights[i]) + biases[i])
    out = sigmoid(np.dot(a, weights[-1]) + biases[-1])
    return out

In the following cell, we define a function to calculate the neural network's ROC AUC (Receiver Operating Characteristic - Area Under the Curve) accuracy. This function takes the actual outcomes (1 for the occurrence of credit default, or 0 otherwise) and the estimated probabilities, and sorts both datasets based on the descending order of the probabilities. Next, the function records the cumulative sum of correctly and incorrectly predicted defaults and converts the values into TPR and FPR (True Positive Rate and False Positive Rate) ratios relative to each final sum. The area under the TPR-FPR curve (AUC) represents the network's accuracy; in the case of a perfect network, the FPR > 0.0 only when TPR = 1.0, which means AUC = 1.0 (a square area), whereas in the case of random guessing, FPR and TPR will grow from 0.0 to 1.0 approximately parallelly, meaning the AUC ≈ 0.5 (a triangular area).

In [22]:
def roc_auc(y_true, y_score):
    desc_order = np.argsort(-y_score.flatten())
    y_true = y_true.flatten()[desc_order]
    y_score = y_score.flatten()[desc_order]
    tps = np.cumsum(y_true)
    fps = np.cumsum(1 - y_true)
    tpr = tps/tps[-1]
    fpr = fps/fps[-1]
    auc = np.trapezoid(tpr, fpr)
    return auc

In the last cell, we finally train the neural network. First, we define hyperparameters such as the number of training epochs, learning rate, dropout rate, and the size of each batch of jointly processed data (batch size). Then, in each epoch, we first shuffle the dataset and execute a forward pass for each batch of samples similarly to the 21st cell (using weights, biases, and sigmoid and ReLU activation functions), while randomly deactivating a certain number of neurons each time for the purpose of better interpreting the relationships between the discussed features. After completing the forward pass, we calculate the cross-entropy loss, which evaluates the difference between the estimated probability of credit default and the actual outcome, and appropriately rewards or penalizes the prediction. Following this, we perform the backpropagation of weights and biases, taking the learning rate into account. Lastly, we calculate the network's accuracy on the training set and on the validation set, and finally save the data.

In [23]:
epochs = 100
lr = 0.02
dr = 0.2
batch_size = 256
n_samples = X_train.shape[0]
loss_history = []
train_auc_history = []
val_auc_history = []

for epoch in range(epochs):
    epoch_loss = 0
    perm = np.random.permutation(n_samples)
    X_shuff = X_train[perm]
    y_shuff = y_train[perm]
    w_shuff = sample_weights[perm]
    
    for i in range(0, n_samples, batch_size):
        X_batch = X_shuff[i:i + batch_size]
        y_batch = y_shuff[i:i + batch_size]
        w_batch = w_shuff[i:i + batch_size]

        activations = [X_batch]
        curr_a = X_batch
        dropout_masks = []
        for j in range(len(weights) - 1):
            curr_a = relu(np.dot(curr_a, weights[j]) + biases[j])
            mask = (np.random.rand(*curr_a.shape) > dr).astype(float)
            curr_a *= mask
            curr_a /= (1 - dr)
            dropout_masks.append(mask)
            activations.append(curr_a)
        out = sigmoid(np.dot(curr_a, weights[-1]) + biases[-1])

        y1 = y_batch*np.log(out + 1e-8)
        y0 = (1 - y_batch)*np.log(1 - out + 1e-8)
        loss = -np.mean(w_batch*(y1 + y0))
        epoch_loss += loss*len(X_batch)

        delta = w_batch*(out - y_batch)/len(X_batch)
        for j in reversed(range(len(weights))):
            dW = np.dot(activations[j].T, delta)
            db = np.sum(delta, axis = 0, keepdims = True)
            if j > 0:
                delta = np.dot(delta, weights[j].T)
                delta *= heaviside(activations[j])
                delta *= dropout_masks[j - 1]
                delta /= (1 - dr)
            weights[j] -= lr*dW
            biases[j] -= lr*db

    epoch_loss /= n_samples
    loss_history.append(epoch_loss)

    train_pred = prob(X_train)
    train_auc = roc_auc(y_train, train_pred)
    train_auc_history.append(train_auc)

    val_pred = prob(X_val)
    val_auc = roc_auc(y_val, val_pred)
    val_auc_history.append(val_auc)